In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, TargetEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score, recall_score, roc_curve
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier
import gc
import os
from itertools import combinations
from tqdm import tqdm
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMRegressor,LGBMClassifier,log_evaluation,early_stopping
try:
    from cuml.ensemble import RandomForestClassifier
    from cuml.neighbors import KNeighborsClassifier
    from cuml.preprocessing import StandardScaler
    from cuml.pipeline import Pipeline
    import cuml
    cuml.set_global_output_type('numpy')
except:
    pass

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

TARGET = 'Irrigation_Need'
NUMS = ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
CATS = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)
SEED_LIST = [60, 0, 2809]
ADD_EXT = True
ALT_FILT = True
MODEL_TYPES = [
                   # 'r',
                   # 'x'
                   # 't'
                   # 'l'
                   # 'h'
                  ]
MODEL_TYPES_2 = [
                   # 'knn',
                   # 'rf',
                   # 'l', 
                   # 't', 
                   # 'r',
                   # 'lr',
                   # 'c', 
                   # 'x'
                   # 't'
                   # 'l'
                   # 'h'
                  ]

def metric(y_true, y_pred):
    y_pred = np.argmax(y_pred, axis=1)
    return balanced_accuracy_score(y_true, y_pred)

def build_score(f):
    f['soil_lt_25'] = 2*(f['Soil_Moisture'] < 25)
    f['rain_lt_300'] = 2*(f['Rainfall_mm'] < 300)
    f['temp_gt_30'] = (f['Temperature_C'] > 30)
    f['wind_gt_10'] = (f['Wind_Speed_kmh'] > 10)
    f['harvest'] = 2*(f['Crop_Growth_Stage'] == 'Harvest')
    f['sowing'] = 2*(f['Crop_Growth_Stage'] == 'Sowing')
    f['mulch'] = (f['Mulching_Used'] == 'Yes')
    f['low_score'] = f['harvest'] + f['sowing'] + f['mulch']
    f['high_score'] = f['soil_lt_25'] + f['rain_lt_300'] + f['temp_gt_30'] + f['wind_gt_10']
    f['score'] = f['high_score'] - f['low_score']
    return f

In [ ]:
PATH = '/kaggle/input/competitions/playground-series-s6e4'
PATH_2 = '/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset'
train = pd.read_csv(f'{PATH}/train.csv', index_col='id')
test = pd.read_csv(f'{PATH}/test.csv', index_col='id')
orig = pd.read_csv(f'{PATH_2}/irrigation_prediction.csv')
train[TARGET] = train[TARGET].map({'Low': 0, 'Medium': 1, 'High': 2})
orig[TARGET] = orig[TARGET].map({'Low': 0, 'Medium': 1, 'High': 2})

M=train[NUMS].max()
def FE(df): 
    
    for c in NUMS:
        for k in range(-4,4):
            df[f"{c}_digit{k}"] = (df[c] // (10**k) % 10).astype('int8')

        if M[c]<10:
            df[c]=df[c].round(3)
        elif M[c]<100:
            df[c]=df[c].round(2)
        else:
            df[c]=df[c].round(1)
    return df 

for df in [train, test, orig]:
    df=FE(df)

In [ ]:
train = build_score(train)
test = build_score(test)
orig = build_score(orig)

In [ ]:
DROP=[c for c in test.columns if test[c].nunique()==1]
print(f"DROP:{DROP}")
train.drop(DROP,axis=1,inplace=True)
test.drop(DROP,axis=1,inplace=True)
orig.drop(DROP,axis=1,inplace=True)

CATEGORY=CATS+[c for c in test.columns if 'digit' in c]
for c in CATEGORY:

    freq = train[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)
    train[c] = train[c].map(lambda x: mapping.get(x, mapping_default))
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default))
    orig[c] = orig[c].map(lambda x: mapping.get(x, mapping_default))
    

FEATURES=CATEGORY+NUMS
train.head()

In [ ]:
TE_columns = []

columns = NUMS + CATS
good_columns = ['Soil_Moisture', 'Crop_Growth_Stage', 'Temperature_C', 'Mulching_Used', 'Wind_Speed_kmh', 'Rainfall_mm']

for r in [2]:
    for cols in tqdm(list(combinations(columns, r))):
        if r == 2 and len(set(cols).intersection(good_columns)) == 0:
            continue
        if r == 3 and len(set(cols).intersection(good_columns)) < 2:
            continue
        name = '-'.join(cols)

        train[name] = train[cols[0]].astype(str)
        for col in cols[1:]:
            train[name] = train[name] + '_' + train[col].astype(str)

        test[name] = test[cols[0]].astype(str)
        for col in cols[1:]:
            test[name] = test[name] + '_' + test[col].astype(str)

        orig[name] = orig[cols[0]].astype(str)
        for col in cols[1:]:
            orig[name] = orig[name] + '_' + orig[col].astype(str)

        combined = pd.concat([train[name], test[name], orig[name]], ignore_index=True)
        combined, _ = combined.factorize()
        if pd.Series(combined).nunique() > len(combined) // 2:
            train = train.drop(name, axis=1)
            test = test.drop(name, axis=1)
            orig = orig.drop(name, axis=1)
            continue
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]
        orig[name] = combined[len(train) + len(test):]

        TE_columns.append(name)

FEATURES = train.columns.tolist()
FEATURES.remove(TARGET)

In [ ]:
cat_cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

num_cols = ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
            'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
            'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']

target_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
reverse_mapping = {0: 'Low', 1: 'Medium', 2: 'High'}
train_full = train.copy()
test_df = test.copy().reset_index(drop=True)
del train, test
gc.collect()
train_full['id'] = train_full.index
train_full[TARGET] = train_full[TARGET].map(reverse_mapping)
submission = pd.read_csv(f'{PATH}/sample_submission.csv')
test_df['id'] = submission['id']

feature_cols = [c for c in train_full.columns if c not in ['id', 'Irrigation_Need'] and train_full[c].dtype != 'object']
feature_cols = cat_cols + [c for c in feature_cols if c not in cat_cols]

print(f'Total features: {len(feature_cols)}')
print(f'Engineered features: {[c for c in feature_cols if c not in cat_cols + num_cols]}')

In [ ]:
y_orig = orig['Irrigation_Need']
y_orig_low = (y_orig == 0).astype('int')
y_orig_med = (y_orig == 1).astype('int')
y_orig_high = (y_orig == 2).astype('int')
orig_target = pd.DataFrame({'low': y_orig_low, 'medium': y_orig_med, 'high': y_orig_high})
X_orig = orig[feature_cols].copy()

In [ ]:
y = train_full['Irrigation_Need'].map(target_mapping).values

for col in cat_cols:
    combined_cats = pd.concat([train_full[col], test_df[col]]).astype(str).unique()
    train_full[col] = pd.Categorical(train_full[col].astype(str), categories=combined_cats)
    test_df[col] = pd.Categorical(test_df[col].astype(str), categories=combined_cats)

features = train_full[feature_cols].copy()
test_features = test_df[feature_cols].copy()
test_ids = test_df['id'].values

for c in cat_cols:
    features[c] = features[c].cat.codes.astype('int32')
    test_features[c] = test_features[c].cat.codes.astype('int32')

In [ ]:
y_low = (y == 0).astype('int')
y_med = (y == 1).astype('int')
y_high = (y == 2).astype('int')
target = pd.DataFrame({'low': y_low, 'medium': y_med, 'high': y_high})

In [ ]:
def to_logits(df):
    df_col = df.columns
    epsilon = 1e-7
    preds_x = np.clip(df, epsilon, 1 - epsilon)
    logits = np.log(preds_x / (1 - preds_x))
    preds_x = pd.DataFrame(logits)
    preds_x.columns = df_col
    return preds_x

def balanced_accuracy():
    def xgb_balanced_accuracy(y_true, y_pred):
        n_classes = 2
        y_pred_labels = np.argmax(y_pred.reshape(-1, n_classes), axis=1)
        return balanced_accuracy_score(y_true.astype(int), y_pred_labels)

    xgb_balanced_accuracy.__name__ = 'bal_ACC'
    return xgb_balanced_accuracy

params_x = {
    'subsample': 0.8747340624732572,
    'colsample_bytree': 0.2,
    'max_depth': 4,
    'learning_rate': 0.02,
    'max_bin': 1024,
    'tree_method': 'hist',
    'device': 'cuda',
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'random_state': 42
}

params_c = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'iterations': 5000,
    'learning_rate': 0.03,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'random_seed': 42,
    'early_stopping_rounds': 1000,
    'task_type': 'GPU',
    'devices': '0'
}

params_l = {'random_state': 60,
             'feature_pre_filter': False,
             'verbose': 100,
             'n_estimators': 20000,
             'learning_rate': 0.05,
             'max_depth': 3,
             'min_child_samples': 63,
             'subsample': 0.812763123433567,
             'colsample_bytree': 0.2029300829885024, 
             'num_leaves': 247, 
             'reg_alpha': 0.07094285437903122,
             'reg_lambda': 0.033039097703242495,
             'max_bin': 32000,
            }

In [ ]:
train_full['score'].value_counts()

In [ ]:
!pip install -q pytabkit

In [ ]:
import random
import torch
import pytabkit
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed=42)

In [ ]:
!pip install -q py-boost

In [ ]:
import os
try:
    # Optional: set the device to run
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    
    os.makedirs('../data', exist_ok=True)
    
    from py_boost import GradientBoosting, SketchBoost
    from py_boost.multioutput.sketching import RandomProjectionSketch, RandomSamplingSketch
except:
    pass

In [ ]:
params_t = {
        'device': 'cuda',
        'val_metric_name': '1-auc_ovr',
        'random_state': 42,
        'verbosity': 0,
        'tabm_k': 32,
        'num_emb_type': 'pwl',
        'd_embedding': 12,
        'batch_size': 256,
        'lr': 3e-3,
        'weight_decay': 2e-2,
        'n_epochs': 3,
        'dropout': 0.1,
        'd_block': 256,
        'n_blocks': 3
    }

In [ ]:
params_r = {
    'random_state': 42,
    'verbosity': 2,
    'val_metric_name': '1-auc_ovr',

    'n_ens': 8,
    'n_epochs': 3,
    'batch_size': 256,
    'use_early_stopping': True,
    'early_stopping_additive_patience': 10,
    'early_stopping_multiplicative_patience': 1,

    'lr': 0.075,
    'wd': 0.0236,
    'sq_mom': 0.988,
    'lr_sched': 'flat_anneal',
    'first_layer_lr_factor': 0.25,
    
    'add_front_scale': False,
    'bias_init_mode': 'neg-uniform-dynamic-2',

    'embedding_size': 6,
    'max_one_hot_cat_size': 18,
    'hidden_sizes': [512, 256, 128],
    'act': 'silu',
    'p_drop': 0.05,
    'p_drop_sched': 'expm4t',

    'plr_hidden_1': 16,
    'plr_hidden_2': 8,
    'plr_act_name': 'gelu',
    'plr_lr_factor': 0.1151,
    'plr_sigma': 2.33,

    'ls_eps': 0.01,
    'ls_eps_sched': 'sqrt_cos',

    'tfms': ['one_hot', 'median_center', 'robust_scale',
             'smooth_clip', 'embedding', 'l2_normalize'],
}

In [ ]:
try:
    params_p = {
        'loss': 'bce',
        'metric': 'auc',
        'ntrees': 7000, 
        'lr': 0.03, 
        'verbose': 100, 
        'es': 300, 
        'lambda_l2': 3, 
        'gd_steps': 1, 
        'subsample': 1, 
        'colsample': 0.2, 
        'min_data_in_leaf': 2,
        'max_bin': 256, 
        'max_depth': 4,
        'seed': 42,
        'multioutput_sketch': RandomProjectionSketch(1)
    }
    
    params_ps = params_p.copy()
    params_ps['multioutput_sketch'] = RandomSamplingSketch(1)
except:
    pass

In [ ]:
params_h = {'max_iter': 20000,
          'random_state': 60,
          'early_stopping': True,
          'categorical_features': "from_dtype",
          'learning_rate': 0.05,
          'loss': 'log_loss',
          'l2_regularization': 1.293878083737259e-05,
          'max_depth': 3, 
          'max_leaf_nodes': 225,
          'min_samples_leaf': 20,
          'tol': 1e-7
         }

In [ ]:
def run(model_type, seed=SEED):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    p_val_x = np.zeros((len(features), 3))
    p_test_x = np.zeros((len(test_features), 3))
    fold_aucs = []
    oof_x = []
    M_x = []
    P_x = []
    V_x = []
    j = 0
    i = 0
    for fold, (tr_idx, val_idx) in enumerate(skf.split(features, target['low'])):
        X_train = features.iloc[tr_idx]
        X_val = features.iloc[val_idx]
        train_target = target.iloc[tr_idx]
        val_target = target.iloc[val_idx]
        y_val = val_target.iloc[:, i]
        val_ids = train_full[['id']].iloc[val_idx].reset_index(drop=True)
        X_test = test_features.copy()
        y_train = train_target.iloc[:, i]

        if model_type == 'lr':
            LOGREG_COLS = ['soil_lt_25', 'temp_gt_30', 'rain_lt_300', 'wind_gt_10', 
                     'harvest', 'sowing', 'mulch', 'low_score', 
                     'high_score', 'score'] + TE_columns
            X_train = X_train[LOGREG_COLS]
            X_val = X_val[LOGREG_COLS]
            X_test = X_test[LOGREG_COLS]
    
        encoder = TargetEncoder(cv=5, random_state=seed, smooth='auto')
        result = pd.DataFrame(encoder.fit_transform(X_train[TE_columns], y_train))
        X_train = pd.concat([result.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
        result = pd.DataFrame(encoder.transform(X_val[TE_columns]))
        X_val = pd.concat([result.reset_index(drop=True), X_val.reset_index(drop=True)], axis=1)
        result = pd.DataFrame(encoder.transform(X_test[TE_columns]))
        X_test = pd.concat([result.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)
        X_train = X_train.drop(TE_columns, axis=1)
        X_val = X_val.drop(TE_columns, axis=1)
        X_test = X_test.drop(TE_columns, axis=1)
    
        X_train.columns = X_train.columns.astype(str)
        X_val.columns = X_val.columns.astype(str)
        X_test.columns = X_test.columns.astype(str)
    
        print(val_target.columns[i])

        if model_type == 't':
            model = TabM_D_Classifier(**params_t)
            model.fit(X_train, train_target.iloc[:, i], X_val, val_target.iloc[:, i])
        elif model_type == 'r':
            model = RealMLP_TD_Classifier(**params_r)
            model.fit(X_train, train_target.iloc[:, i], X_val, val_target.iloc[:, i])
        elif model_type == 'x':
            model = XGBClassifier(
                **params_x,
                n_estimators=5000,
                early_stopping_rounds=500,
            )

            model.fit(
                X_train, train_target.iloc[:, i],
                eval_set=[(X_val, val_target.iloc[:, i])],
                verbose=1000
            )
        elif model_type == 'c':
            model = CatBoostClassifier(**params_c)
    
            model.fit(
                X_train, train_target.iloc[:, i],
                eval_set=(X_val, val_target.iloc[:, i]),
                verbose=500
            )
        elif model_type == 'lr':
            model = LogisticRegressionCV(max_iter=10000, random_state=42, Cs=10, scoring='roc_auc', cv=5)

            model.fit(
                X_train, train_target.iloc[:, i]
            )
        elif model_type == 'l':
            model=LGBMClassifier(**params_l)
    
            model.fit(X_train, y_train,eval_set=[(X_val, y_val)],
                      eval_metric='auc',
                      callbacks=[log_evaluation(100),early_stopping(250)])
        elif model_type == 'rf':
            model = RandomForestClassifier(n_estimators=500, max_depth=12, random_state=42, max_features=0.3, output_type='numpy')

            model.fit(
                X_train, train_target.iloc[:, i]
            )
        elif model_type == 'knn':
            X_train = X_train.astype('float32')
            X_val = X_val.astype('float32')
            X_test = X_test.astype('float32')
            model = Pipeline([
                ('scaler', StandardScaler()), 
                ('knn', KNeighborsClassifier(n_neighbors=200, weights='distance'))
            ])
            model.fit(X_train, train_target.iloc[:, i])
        elif model_type == 'p':
            model = GradientBoosting(**params_p)
            model.fit(X_train,train_target.iloc[:, i],eval_sets=[{'X': X_val, 'y': val_target.iloc[:, i]}],
                     )
        elif model_type == 'ps':
            model = GradientBoosting(**params_ps)
            model.fit(X_train,train_target.iloc[:, i],eval_sets=[{'X': X_val, 'y': val_target.iloc[:, i]}],
                     )
        elif model_type == 'h':
            model = HistGradientBoostingClassifier(**params_h)
            model.fit(X_train, train_target.iloc[:, i], X_val=X_val, y_val=val_target.iloc[:, i])
                        
        if model_type not in ['p', 'ps']:
            p_val = model.predict_proba(X_val)[:, 1]
            p_test = model.predict_proba(X_test)[:, 1]
        else:
            p_val = model.predict(X_val)
            p_test = model.predict(X_test)
        p_val_x[val_idx, i] = p_val
        p_test_x[:, i] += p_test / N_FOLDS
    
        val_metric_x = roc_auc_score(val_target.iloc[:, i], p_val_x[val_idx, i])
        print(val_metric_x)
        V_x.append(val_metric_x)
    
        p_val_x_2 = pd.DataFrame(p_val_x[val_idx, :], columns=target.columns)
        p_val_x_2['id'] = val_ids
        oof_x.append(p_val_x_2)
        M_x.append(val_metric_x)
    
        p_test_x_2 = pd.DataFrame(p_test_x)
        P_x.append(p_test_x_2)
        if j < 4: del model
        del X_train, X_val, y_train, y_val
        gc.collect()
        j += 1
    
    oof_preds_t = pd.concat(oof_x, ignore_index=True).sort_values(by='id').reset_index(drop=True)
    oof_preds_t.to_parquet(f'oof_preds_{model_type}_{seed}.parquet')
    print(M_x)
    print(np.mean(M_x))
    p_test_x_final = np.mean(P_x, axis=0)
    preds_final_t = pd.DataFrame(p_test_x_final, columns=target.columns)
    preds_final_t['id'] = test_ids
    preds_final_t.to_parquet(f'submission_{model_type}_{seed}.parquet')
    h = pd.DataFrame(target.sum(), columns=['count']).astype(int)
    h['auc_val'] = np.round(np.mean(V_x, axis=0), 6)
    return h

In [ ]:
add_oof = [
    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_final.csv'),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/mahog/pg-s6e4-realmlp-cv-0-97802-lb-0-97685/realmlp_oof.csv"
    ).values.reshape(630_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/mahog/pg-s6e4-xgb-cv-0-97723-lb-0-97644/xgb_oof.csv"
    ).values.reshape(630_000, 3)),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/utaazu/0-979-cv-single-cat-pairwise-te-bias-tuning/oof_probs_cat.npy"
    )),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/utaazu/s6e4-eda-decoding-synthetic-bias-optimized-lgbm/oof_probs_lgbm.npy"
    )),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yunsuxiaozi/pss6e4-lgb-baselinecv-0-97943/oof_preds.npy"
    )),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yunsuxiaozi/pss6e4-lgb-advanced-cv-0-97997/oof_preds.npy"
    )),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yunsuxiaozi/pss6e4-xgb-cv-0-979805/oof_preds.npy"
    )),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/include4eto/ps6e4-xgb-cudf-pseudo-labels/001_oof.csv"
    ).values.reshape(630_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/include4eto/ps6e4-tab-transformer-claude-vibe-coding/002_oof.csv"
    ).values.reshape(630_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/include4eto/ps6e4-one-vs-rest-xgb-claude-vibe-coding/001_oof.csv"
    ).values.reshape(630_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yekenot/ps-s6-e4-trompt-pytorch-frame/oof_preds.csv"
    ).values.reshape(630_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yekenot/ps-s6-e4-realmlp-pytabkit/oof_preds.csv"
    ).values.reshape(630_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds/oof_preds_seed_60.csv"
    ).values.reshape(630_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yekenot/ps-s6-e4-realmlp-slightly-polished/oof_preds.csv"
    ).values.reshape(630_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yekenot/ps-s6-e4-realmlp-slightly-polished/oof_preds_v2.csv"
    ).values.reshape(630_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yekenot/ps-s6-e4-realmlp-slightly-polished/oof_preds_v3.csv"
    ).values.reshape(630_000, 3)),

     pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds/oof_preds_l_seed_60.csv"
    ).values.reshape(630_000, 3)),

    pd.read_csv(f'/kaggle/input/notebooks/kirill0212/irrigation-need-catboost/CAT2_8_oof.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/LGBM2_9_oof.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/HGB2_oof.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/Realmlp2_3_oof.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/YDF_GBT_oof.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/LR2_7_oof.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/NN_4_oof.csv'),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds/oof_gnn.npy"
    )),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/oof_probs_final.npy"
    )),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_preds_xgb_only.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_preds_rmlp_only.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_preds_tabm_only.csv'),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/oof_preds.npy"
    )),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_pred_xgb.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_pred_lgb.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_pred_rmlp.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_pred_xgb_2.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_pred_tabm_2.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_pred_lgb_seed_60.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_final_r.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_final_x.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_final_l.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_final_h.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_final_rf.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_final_x_dart.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/oof_final_t.csv'),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds/oof_preds_tabm_seed_60.csv"
    ).values.reshape(630_000, 3)),
]

In [ ]:
add_test = [
    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_final.csv'),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/mahog/pg-s6e4-realmlp-cv-0-97802-lb-0-97685/realmlp_pred.csv",
    ).values.reshape(270_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/mahog/pg-s6e4-xgb-cv-0-97723-lb-0-97644/xgb_pred.csv",
    ).values.reshape(270_000, 3)),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/utaazu/0-979-cv-single-cat-pairwise-te-bias-tuning/test_probs_cat.npy"
    )),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/utaazu/s6e4-eda-decoding-synthetic-bias-optimized-lgbm/test_probs_lgbm.npy"
    )),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yunsuxiaozi/pss6e4-lgb-baselinecv-0-97943/test_preds.npy"
    )),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yunsuxiaozi/pss6e4-lgb-advanced-cv-0-97997/test_preds.npy"
    )),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yunsuxiaozi/pss6e4-xgb-cv-0-979805/test_preds.npy"
    )),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/include4eto/ps6e4-xgb-cudf-pseudo-labels/001_test.csv"
    ).values.reshape(270_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/include4eto/ps6e4-tab-transformer-claude-vibe-coding/002_test.csv"
    ).values.reshape(270_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/include4eto/ps6e4-one-vs-rest-xgb-claude-vibe-coding/001_test.csv"
    ).values.reshape(270_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yekenot/ps-s6-e4-trompt-pytorch-frame/test_preds.csv"
    ).values.reshape(270_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yekenot/ps-s6-e4-realmlp-pytabkit/test_preds.csv"
    ).values.reshape(270_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds/test_preds_seed_60.csv"
    ).values.reshape(270_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yekenot/ps-s6-e4-realmlp-slightly-polished/test_preds.csv"
    ).values.reshape(270_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yekenot/ps-s6-e4-realmlp-slightly-polished/test_preds_v2.csv"
    ).values.reshape(270_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/a/yekenot/ps-s6-e4-realmlp-slightly-polished/test_preds_v3.csv"
    ).values.reshape(270_000, 3)),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds/test_preds_l_seed_60.csv"
    ).values.reshape(270_000, 3)),

    pd.read_csv(f'/kaggle/input/notebooks/kirill0212/irrigation-need-catboost/CAT2_8_test.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/LGBM2_9_test.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/HGB2_test.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/Realmlp2_3_test.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/YDF_GBT_test.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/LR2_7_test.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/NN_4_test.csv'),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds/test_gnn.npy"
    )),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/test_probs_final.npy"
    )),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_preds_xgb_only.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_preds_rmlp_only.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_preds_tabm_only.csv'),

    pd.DataFrame(np.load(
        f"/kaggle/input/datasets/kirill0212/my-preds-4/test_preds.npy"
    )),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_pred_xgb.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_pred_lgb.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_pred_rmlp.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_pred_xgb_2.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_pred_tabm_2.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_pred_lgb_seed_60.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_final_r.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_final_x.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_final_l.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_final_h.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_final_rf.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_final_x_dart.csv'),

    pd.read_csv(f'/kaggle/input/datasets/kirill0212/my-preds/test_final_t.csv'),

    pd.DataFrame(pd.read_csv(
        f"/kaggle/input/datasets/kirill0212/my-preds/test_preds_tabm_seed_60.csv"
    ).values.reshape(270_000, 3)),
]

In [ ]:
add_features = [
    'my_rmlp_xgb_2',
    'mahog_rmlp',
    'mahog_xgb',
    'utaazu_cat',
    'utaazu_lgb',
    'yunsuxiaozi_lgb',
    'yunsuxiaozi_lgb_a',
    'yunsuxiaozi_xgb',
    'include4eto_xgb',
    'include4eto_trans',
    'include4eto_xgb_2',
    'yekenot_trompt',
    'yekenot_rmlp',
    'yekenot_rmlp_seed_60',
    'yekenot_rmlp_polished',
    'yekenot_rmlp_polished_v2',
    'yekenot_rmlp_polished_v3',
    'my_lgb_seed_60',
    'mikhail_naumov_cat',
    'mikhail_naumov_lgb',
    'mikhail_naumov_hgb',
    'mikhail_naumov_rmlp',
    'mikhail_naumov_ydf',
    'mikhail_naumov_lr',
    'mikhail_naumov_nn',
    'gnn',
    'maria_nadeem_lgb',
    'lucas_moraes_xgb',
    'lucas_moraes_rmlp',
    'lucas_moraes_tabm',
    'rohit_lgb',
    'ashish_xgb',
    'ashish_lgb',
    'ashish_rmlp',
    'ashish_xgb_2',
    'ashish_tabm_2',
    'ashish_lgb_seed_60',
    'my_rmlp',
    'my_xgb',
    'my_lgb',
    'my_hgb',
    'my_rf',
    'my_xgb_dart',
    'my_tabm',
    'my_tabm_seed_60',
]

In [ ]:
all_files = os.listdir(
    f"/kaggle/input/datasets/wguesdon/ps6e4-30model-oof-predictions"
)

all_files = [c for c in all_files if c.startswith("oof")]
for myfile in all_files:
    df = np.load(f"/kaggle/input/datasets/wguesdon/ps6e4-30model-oof-predictions/{myfile}")[:, [1,2,0]]
    add_oof.append(pd.DataFrame(df))
    label = myfile.replace("oof_", "pred_")
    df_2 = np.load(f"/kaggle/input/datasets/wguesdon/ps6e4-30model-oof-predictions/{label}")[:, [1,2,0]]
    add_test.append(pd.DataFrame(df_2))
    add_features.append('wguesdon_' + myfile.split('.')[0][4:])
print(all_files)

In [ ]:
# these were excluded in best scoring sub
exclude = [
    'yekenot_rmlp_polished',
    'yekenot_rmlp_polished_v2',
    'yekenot_rmlp_polished_v3',
    'my_lgb_seed_60',
    'maria_nadeem_lgb',
    'lucas_moraes_xgb',
    'lucas_moraes_rmlp',
    'lucas_moraes_tabm',
    'ashish_lgb',
    'ashish_xgb_2',
    'ashish_tabm_2',
    'ashish_lgb_seed_60',
    'my_lgb',
    'my_hgb',
]

exclude_ind = [i for i, val in enumerate(add_features) if val in exclude]

add_oof = [v for i, v in enumerate(add_oof) if i not in exclude_ind]
add_test = [v for i, v in enumerate(add_test) if i not in exclude_ind]
add_features = [v for i, v in enumerate(add_features) if i not in exclude_ind]
L = len(add_oof)

In [ ]:
def public_preds(proba, bias):
    return np.argmax(np.log(np.clip(proba, 1e-15, 1.0)) + bias, axis=1)

def tune_bias(proba, y_true):
    best_bias = np.zeros(proba.shape[1], dtype=np.float64)
    best_score = balanced_accuracy_score(y_true, public_preds(proba, best_bias))
    history = [best_score]
    
    for step in (1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002):
        improved = True
        while improved:
            improved = False
            for ci in range(proba.shape[1]):
                for d in (-1.0, 1.0):
                    c = best_bias.copy()
                    c[ci] += d * step
                    s = balanced_accuracy_score(y_true, public_preds(proba, c))
                    if s > best_score + 1e-9:
                        best_bias, best_score, improved = c, s, True
                        history.append(best_score)
    return best_bias, best_score, history

In [ ]:
oof_acc = pd.DataFrame(columns=['model', 'score'])
for i in tqdm(range(L)):
    best_bias, tuned_ba, opt_history = tune_bias(add_oof[i], y)
    oof_acc.loc[i] = [add_features[i], tuned_ba]
print(oof_acc.sort_values(by='score'))

In [ ]:
oof_auc_low = pd.DataFrame(columns=['model', 'score'])
for i in tqdm(range(L)):
    auc_low = roc_auc_score(y_low, add_oof[i].iloc[:, 0])
    oof_auc_low.loc[i] = [add_features[i], auc_low]
print(oof_auc_low.sort_values(by='score'))

In [ ]:
oof_auc_med = pd.DataFrame(columns=['model', 'score'])
for i in tqdm(range(L)):
    auc_med = roc_auc_score(y_med, add_oof[i].iloc[:, 1])
    oof_auc_med.loc[i] = [add_features[i], auc_med]
print(oof_auc_med.sort_values(by='score'))

In [ ]:
oof_auc_high = pd.DataFrame(columns=['model', 'score'])
for i in tqdm(range(L)):
    auc_high = roc_auc_score(y_high, add_oof[i].iloc[:, 2])
    oof_auc_high.loc[i] = [add_features[i], auc_high]
print(oof_auc_high.sort_values(by='score'))

In [ ]:
print(pd.Series(add_features))

In [ ]:
s = {f'{i}': add_oof[i].iloc[:, 0] for i in range(L)}
pd.DataFrame(s).corr()

In [ ]:
pd.DataFrame(s).max(), pd.DataFrame(s).min()

In [ ]:
s = {f'{i}': add_test[i].iloc[:, 0] for i in range(L)}
pd.DataFrame(s).corr()

In [ ]:
s = {f'{i}': add_oof[i].iloc[:, 2] for i in range(L)}
pd.DataFrame(s).corr()

In [ ]:
s = {f'{i}': add_test[i].iloc[:, 2] for i in range(L)}
pd.DataFrame(s).corr()

In [ ]:
s = {f'{i}': add_oof[i].iloc[:, 1] for i in range(L)}
pd.DataFrame(s).corr()

In [ ]:
s = {f'{i}': add_test[i].iloc[:, 1] for i in range(L)}
pd.DataFrame(s).corr()

In [ ]:
preds_tmp = train_full[['id']]
preds_l = []
test_l = []
pred_col = []
for model_type in MODEL_TYPES:
    for seed in SEED_LIST:
        run(model_type, seed)
        preds_l.append(pd.read_parquet(f'oof_preds_{model_type}_{seed}.parquet')[['low']].rename(columns={'low': f'my_base_{model_type}_seed_{seed}'}))
        test_l.append(pd.read_parquet(f'submission_{model_type}_{seed}.parquet')[['low']].rename(columns={'low': f'my_base_{model_type}_seed_{seed}'}))
        pred_col.append(f'my_base_{model_type}_seed_{seed}')

if ADD_EXT:
    cnt = 0
    for oof_tmp in add_oof:
        preds_l.append(oof_tmp.iloc[:, 0])
        pred_col.append(add_features[cnt])
        cnt += 1
    for test_tmp in add_test:
        test_l.append(test_tmp.iloc[:, 0])
preds_tmp = pd.concat(
                    preds_l
, axis=1)
test_tmp = pd.concat(
                    test_l
, axis=1)
preds_tmp = to_logits(preds_tmp)
test_tmp = to_logits(test_tmp)
preds_tmp.columns = pred_col
test_tmp.columns = pred_col
print(preds_tmp.shape, test_tmp.shape, preds_tmp.mean(), test_tmp.mean(), preds_tmp.max(), test_tmp.max(), preds_tmp.min(), test_tmp.min())

In [ ]:
def run_blend(seed=SEED):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    p_val_x = np.zeros((len(features), 3))
    p_test_x = np.zeros((len(test_features), 3))
    fold_aucs = []
    oof_x = []
    M_x = []
    P_x = []
    V_x = []
    j = 0
    i = 0
    for fold, (tr_idx, val_idx) in tqdm(enumerate(skf.split(preds_tmp, target['low']))):
        X_train = preds_tmp.iloc[tr_idx]
        X_val = preds_tmp.iloc[val_idx]
        train_target = target.iloc[tr_idx]
        val_target = target.iloc[val_idx]
        y_train = train_target.iloc[:, i]
        y_val = val_target.iloc[:, i]
        val_ids = train_full[['id']].iloc[val_idx].reset_index(drop=True)
        X_test = test_tmp.copy()
    
        X_train.columns = X_train.columns.astype(str)
        X_val.columns = X_val.columns.astype(str)
        X_test.columns = X_test.columns.astype(str)
    
        print(val_target.columns[i])
        
        model = LogisticRegressionCV(max_iter=1000, random_state=seed, Cs=10, scoring='roc_auc', cv=5)
    
        model.fit(
            X_train, train_target.iloc[:, i]
        )
    
        p_val = model.predict_proba(X_val)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]
        p_val_x[val_idx, i] = p_val
        p_test_x[:, i] += p_test / N_FOLDS
    
        val_metric_x = roc_auc_score(val_target.iloc[:, i], p_val_x[val_idx, i])
        print(val_metric_x)
        V_x.append(val_metric_x)
    
        p_val_x_2 = pd.DataFrame(p_val_x[val_idx, :], columns=target.columns)
        p_val_x_2['id'] = val_ids
        oof_x.append(p_val_x_2)
        M_x.append(val_metric_x)
    
        p_test_x_2 = pd.DataFrame(p_test_x)
        P_x.append(p_test_x_2)
        map_coef = pd.Series(model.coef_[0], index=model.feature_names_in_)
        print(map_coef.sort_values())
        if j < 4: del model
        del X_train, X_val, y_train, y_val
        gc.collect()
        j += 1
    
    oof_preds = pd.concat(oof_x, ignore_index=True).sort_values(by='id').reset_index(drop=True)
    oof_preds.to_parquet('oof_preds_blend.parquet')
    print(M_x)
    print(np.mean(M_x))
    p_test_x_final = np.mean(P_x, axis=0)
    preds_final = pd.DataFrame(p_test_x_final, columns=target.columns)
    preds_final['id'] = test_ids
    preds_final.to_parquet(f'submission_blend.parquet')
    h = pd.DataFrame(target.sum(), columns=['count']).astype(int)
    h['auc_val'] = np.round(np.mean(V_x, axis=0), 6)
    return oof_preds, preds_final

In [ ]:
oof_x = []
P_x = []
for seed in SEED_LIST:
    oof_preds_tmp, preds_final_tmp = run_blend(seed)
    oof_x.append(oof_preds_tmp.set_index('id'))
    P_x.append(preds_final_tmp.set_index('id'))
    train_ids = oof_preds_tmp[['id']]
oof_preds = pd.DataFrame(np.mean(oof_x, axis=0))
oof_preds.columns = target.columns
oof_preds['id'] = train_ids['id'].copy()
preds_final = pd.DataFrame(np.mean(P_x, axis=0))
preds_final.columns = target.columns
preds_final['id'] = test_ids

In [ ]:
fpr, tpr, thresholds = roc_curve(target['low'], oof_preds['low'])
balanced_accuracies = (tpr + (1 - fpr)) / 2
best_threshold = thresholds[np.argmax(balanced_accuracies)]
print(f"Optimal Threshold from OOF: {best_threshold}")
oof_preds_binary = (oof_preds['low'] >= best_threshold).astype(int)
rec0 = recall_score(target['low'], oof_preds_binary, pos_label=0)
rec1 = recall_score(target['low'], oof_preds_binary, pos_label=1)
score = balanced_accuracy_score(target['low'], oof_preds_binary)
print(f"Recall Class 0: {rec0:.6f}")
print(f"Recall Class 1: {rec1:.6f}")
print(f"Balanced Accuracy: {score:.6f}")
test_preds_binary = (preds_final['low'] >= best_threshold).astype(int)

In [ ]:
def run_2(model_type, seed=SEED):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    p_val_x = np.zeros((len(features), 3))
    p_test_x = np.zeros((len(test_features), 3))
    fold_aucs = []
    oof_x = []
    M_x = []
    P_x = []
    V_x = []
    j = 0
    i = 2
    for fold, (tr_idx, val_idx) in enumerate(skf.split(features, target['high'])):
        X_train = features.iloc[tr_idx].reset_index(drop=True)
        X_val = features.iloc[val_idx].reset_index(drop=True)
        X_val_full = features.iloc[val_idx].copy().reset_index(drop=True)
        train_target = target.iloc[tr_idx].reset_index(drop=True)
        val_target = target.iloc[val_idx].reset_index(drop=True)
        # filtering train and val!
        if ALT_FILT == False:
            train_mask_2 = (train_target['low'] == 0)
            val_mask_2 = (val_target['low'] == 0)
            X_train = X_train[train_mask_2]
            train_target = train_target[train_mask_2]
            X_val = X_val[val_mask_2]
            val_target = val_target[val_mask_2]
        else:
            # alternative 
            train_mask_2 = (oof_preds_binary[tr_idx] == 0).values
            val_mask_2 = (oof_preds_binary[val_idx] == 0).values
            X_train = X_train[train_mask_2]
            train_target = train_target[train_mask_2]
            X_val = X_val[val_mask_2]
            val_target = val_target[val_mask_2]
 
        val_ids = train_full[['id']].iloc[val_idx].reset_index(drop=True)
        y_train = train_target.iloc[:, i]
        y_val = val_target.iloc[:, i]
        X_test = test_features.copy()

        if model_type == 'lr':
            LOGREG_COLS = ['soil_lt_25', 'temp_gt_30', 'rain_lt_300', 'wind_gt_10', 
                     'harvest', 'sowing', 'mulch', 'low_score', 
                     'high_score', 'score'] + TE_columns
            X_train = X_train[LOGREG_COLS]
            X_val = X_val[LOGREG_COLS]
            X_test = X_test[LOGREG_COLS]
    
        encoder = TargetEncoder(cv=5, random_state=seed, smooth='auto')
        result = pd.DataFrame(encoder.fit_transform(X_train[TE_columns], y_train))
        X_train = pd.concat([result.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
        result = pd.DataFrame(encoder.transform(X_val[TE_columns]))
        X_val = pd.concat([result.reset_index(drop=True), X_val.reset_index(drop=True)], axis=1)
        result = pd.DataFrame(encoder.transform(X_val_full[TE_columns]))
        X_val_full = pd.concat([result.reset_index(drop=True), X_val_full.reset_index(drop=True)], axis=1)
        result = pd.DataFrame(encoder.transform(X_test[TE_columns]))
        X_test = pd.concat([result.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)
        X_train = X_train.drop(TE_columns, axis=1)
        X_val = X_val.drop(TE_columns, axis=1)
        X_val_full = X_val_full.drop(TE_columns, axis=1)
        X_test = X_test.drop(TE_columns, axis=1)
    
        X_train.columns = X_train.columns.astype(str)
        X_val.columns = X_val.columns.astype(str)
        X_val_full.columns = X_val_full.columns.astype(str)
        X_test.columns = X_test.columns.astype(str)
    
        print(val_target.columns[i])
        
        if model_type == 't':
            model = TabM_D_Classifier(**params_t)
            model.fit(X_train, train_target.iloc[:, i], X_val, val_target.iloc[:, i])
        elif model_type == 'r':
            model = RealMLP_TD_Classifier(**params_r)
            model.fit(X_train, train_target.iloc[:, i], X_val, val_target.iloc[:, i])
        elif model_type == 'x':
            model = XGBClassifier(
                **params_x,
                n_estimators=5000,
                early_stopping_rounds=500,
            )

            model.fit(
                X_train, train_target.iloc[:, i],
                eval_set=[(X_val, val_target.iloc[:, i])],
                verbose=1000
            )
        elif model_type == 'c':
            model = CatBoostClassifier(**params_c)
    
            model.fit(
                X_train, train_target.iloc[:, i],
                eval_set=(X_val, val_target.iloc[:, i]),
                verbose=500
            )
        elif model_type == 'lr':
            model = LogisticRegressionCV(max_iter=10000, random_state=42, Cs=10, scoring='roc_auc', cv=5)

            model.fit(
                X_train, train_target.iloc[:, i]
            )
        elif model_type == 'l':
            model=LGBMClassifier(**params_l)
    
            model.fit(X_train, y_train,eval_set=[(X_val, y_val)],
                      eval_metric='auc',
                      callbacks=[log_evaluation(100),early_stopping(250)])
        elif model_type == 'rf':
            model = RandomForestClassifier(n_estimators=500, max_depth=12, random_state=42, max_features=0.3, output_type='numpy')

            model.fit(
                X_train, train_target.iloc[:, i]
            )
        elif model_type == 'knn':
            X_train = X_train.astype('float32')
            X_val = X_val.astype('float32')
            X_test = X_test.astype('float32')
            model = Pipeline([
                ('scaler', StandardScaler()), 
                ('knn', KNeighborsClassifier(n_neighbors=200, weights='distance'))
            ])
            model.fit(X_train, train_target.iloc[:, i])
        elif model_type == 'p':
            model = GradientBoosting(**params_p)
            model.fit(X_train,train_target.iloc[:, i],eval_sets=[{'X': X_val, 'y': val_target.iloc[:, i]}],
                     )
        elif model_type == 'ps':
            model = GradientBoosting(**params_ps)
            model.fit(X_train,train_target.iloc[:, i],eval_sets=[{'X': X_val, 'y': val_target.iloc[:, i]}],
                     )
        elif model_type == 'h':
            model = HistGradientBoostingClassifier(**params_h)
            model.fit(X_train, train_target.iloc[:, i], X_val=X_val, y_val=val_target.iloc[:, i])

        if model_type not in ['p', 'ps']:
            p_val = model.predict_proba(X_val)[:, 1]
            p_val_full = model.predict_proba(X_val_full)[:, 1]
            p_test = model.predict_proba(X_test)[:, 1]
        else:
            p_val = model.predict(X_val)
            p_val_full = model.predict(X_val_full)
            p_test = model.predict(X_test)
        p_val_x[val_idx, i] = p_val_full
        p_test_x[:, i] += p_test / N_FOLDS
    
        val_metric_x = roc_auc_score(val_target.iloc[:, i], p_val)
        print(val_metric_x)
        V_x.append(val_metric_x)
    
        p_val_x_2 = pd.DataFrame(p_val_x[val_idx, :], columns=target.columns)
        p_val_x_2['id'] = val_ids
        oof_x.append(p_val_x_2)
        M_x.append(val_metric_x)
    
        p_test_x_2 = pd.DataFrame(p_test_x)
        P_x.append(p_test_x_2)
        if j < 4: del model
        del X_train, X_val, y_train, y_val
        gc.collect()
        j += 1
    
    oof_preds_2_t = pd.concat(oof_x, ignore_index=True).sort_values(by='id').reset_index(drop=True)
    oof_preds_2_t.to_parquet(f'oof_preds_2_{model_type}_{seed}.parquet')
    print(M_x)
    print(np.mean(M_x))
    p_test_x_final = np.mean(P_x, axis=0)
    preds_final_2_t = pd.DataFrame(p_test_x_final, columns=target.columns)
    preds_final_2_t['id'] = test_ids
    preds_final_2_t.to_parquet(f'submission_2_{model_type}_{seed}.parquet')
    h = pd.DataFrame(target.sum(), columns=['count']).astype(int)
    h['auc_val'] = np.round(np.mean(V_x, axis=0), 6)
    return h

In [ ]:
preds_tmp_2 = train_full[['id']]
preds_l_2 = []
test_l_2 = []
pred_col_2 = []
for model_type in MODEL_TYPES_2:
    for seed in SEED_LIST:
        run_2(model_type, seed)
        preds_l_2.append(pd.read_parquet(f'oof_preds_2_{model_type}_{seed}.parquet')[['high']].rename(columns={'high': f'my_base_{model_type}_seed_{seed}'}))
        test_l_2.append(pd.read_parquet(f'submission_2_{model_type}_{seed}.parquet')[['high']].rename(columns={'high': f'my_base_{model_type}_seed_{seed}'}))
        pred_col_2.append(f'my_base_{model_type}_seed_{seed}')
if ADD_EXT:
    cnt_2 = 0
    for oof_tmp in add_oof:
        preds_l_2.append(oof_tmp.iloc[:, 2])
        pred_col_2.append(add_features[cnt_2])
        cnt_2 += 1
    for test_tmp in add_test:
        test_l_2.append(test_tmp.iloc[:, 2])
preds_tmp_2 = pd.concat(
                    preds_l_2
, axis=1)
test_tmp_2 = pd.concat(
                    test_l_2
, axis=1)
preds_tmp_2 = to_logits(preds_tmp_2)
test_tmp_2 = to_logits(test_tmp_2)
preds_tmp_2.columns = pred_col_2
test_tmp_2.columns = pred_col_2
print(preds_tmp_2.shape, test_tmp_2.shape, preds_tmp_2.mean(), test_tmp_2.mean(), preds_tmp_2.max(), test_tmp_2.max(), preds_tmp_2.min(), test_tmp_2.min())

In [ ]:
def run_blend_2(seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    p_val_x = np.zeros((len(features), 3))
    p_test_x = np.zeros((len(test_features), 3))
    fold_aucs = []
    oof_x = []
    M_x = []
    P_x = []
    V_x = []
    j = 0
    i = 2
    for fold, (tr_idx, val_idx) in tqdm(enumerate(skf.split(preds_tmp_2, target['high']))):
        X_train = preds_tmp_2.iloc[tr_idx].reset_index(drop=True)
        X_val = preds_tmp_2.iloc[val_idx].reset_index(drop=True)
        X_val_full = preds_tmp_2.iloc[val_idx].copy().reset_index(drop=True)
        train_target = target.iloc[tr_idx].reset_index(drop=True)
        val_target = target.iloc[val_idx].reset_index(drop=True)
        # filtering train and val!
        if ALT_FILT == False:
            train_mask_2 = (train_target['low'] == 0)
            val_mask_2 = (val_target['low'] == 0)
            X_train = X_train[train_mask_2]
            train_target = train_target[train_mask_2]
            X_val = X_val[val_mask_2]
            val_target = val_target[val_mask_2]
        else:
            # alternative 
            train_mask_2 = (oof_preds_binary[tr_idx] == 0).values
            val_mask_2 = (oof_preds_binary[val_idx] == 0).values
            X_train = X_train[train_mask_2]
            train_target = train_target[train_mask_2]
            X_val = X_val[val_mask_2]
            val_target = val_target[val_mask_2]
    
        val_ids = train_full[['id']].iloc[val_idx].reset_index(drop=True)
        y_train = train_target.iloc[:, i]
        y_val = val_target.iloc[:, i]
        X_test = test_tmp_2.copy()
    
        X_train.columns = X_train.columns.astype(str)
        X_val.columns = X_val.columns.astype(str)
        X_val_full.columns = X_val_full.columns.astype(str)
        X_test.columns = X_test.columns.astype(str)
    
        print(val_target.columns[i])
        
        model = LogisticRegressionCV(max_iter=1000, random_state=seed, Cs=10, scoring='roc_auc', cv=5)
    
        model.fit(
            X_train, train_target.iloc[:, i]
        )
    
        p_val = model.predict_proba(X_val)[:, 1]
        p_val_full = model.predict_proba(X_val_full)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]
        p_val_x[val_idx, i] = p_val_full
        p_test_x[:, i] += p_test / N_FOLDS
    
        val_metric_x = roc_auc_score(val_target.iloc[:, i], p_val)
        print(val_metric_x)
        V_x.append(val_metric_x)
    
        p_val_x_2 = pd.DataFrame(p_val_x[val_idx, :], columns=target.columns)
        p_val_x_2['id'] = val_ids
        oof_x.append(p_val_x_2)
        M_x.append(val_metric_x)
    
        p_test_x_2 = pd.DataFrame(p_test_x)
        P_x.append(p_test_x_2)
        map_coef = pd.Series(model.coef_[0], index=model.feature_names_in_)
        print(map_coef.sort_values())
        if j < 4: del model
        del X_train, X_val, y_train, y_val
        gc.collect()
        j += 1
    
    oof_preds_2 = pd.concat(oof_x, ignore_index=True).sort_values(by='id').reset_index(drop=True)
    oof_preds_2.to_parquet('oof_preds_2_blend.parquet')
    print(M_x)
    print(np.mean(M_x))
    p_test_x_final = np.mean(P_x, axis=0)
    preds_final_2 = pd.DataFrame(p_test_x_final, columns=target.columns)
    preds_final_2['id'] = test_ids
    preds_final_2.to_parquet(f'submission_2_blend.parquet')
    h = pd.DataFrame(target.sum(), columns=['count']).astype(int)
    h['auc_val'] = np.round(np.mean(V_x, axis=0), 6)
    return oof_preds_2, preds_final_2

In [ ]:
oof_x = []
P_x = []
for seed in SEED_LIST:
    oof_preds_tmp, preds_final_tmp = run_blend_2(seed)
    oof_x.append(oof_preds_tmp.set_index('id'))
    P_x.append(preds_final_tmp.set_index('id'))
    train_ids = oof_preds_tmp[['id']]
oof_preds_2 = pd.DataFrame(np.mean(oof_x, axis=0))
oof_preds_2.columns = target.columns
oof_preds_2['id'] = train_ids['id'].copy()
preds_final_2 = pd.DataFrame(np.mean(P_x, axis=0))
preds_final_2.columns = target.columns
preds_final_2['id'] = test_ids

In [ ]:
fpr, tpr, thresholds = roc_curve(target['high'], oof_preds_2['high'])
recall_0 = 1 - fpr
recall_1 = tpr
weighted_scores = (0.7 * recall_0 + 0.3 * recall_1)
best_idx = np.argmax(weighted_scores)
best_threshold = thresholds[best_idx]
print(f"Optimal Threshold from OOF: {best_threshold}")
oof_preds_binary_2 = (oof_preds_2['high'] >= best_threshold).astype(int)
rec0 = recall_score(target['high'], oof_preds_binary_2, pos_label=0)
rec1 = recall_score(target['high'], oof_preds_binary_2, pos_label=1)
score = balanced_accuracy_score(target['high'], oof_preds_binary_2)
print(f"Recall Class 0: {rec0:.6f}")
print(f"Recall Class 1: {rec1:.6f}")
print(f"Balanced Accuracy: {score:.6f}")
test_preds_binary_2 = (preds_final_2['high'] >= best_threshold).astype(int)

oof_final = oof_preds.copy()
oof_final['Irrigation_Need'] = 'Medium'
oof_final.loc[oof_preds_binary == 1, 'Irrigation_Need'] = 'Low'
oof_final[(oof_final['Irrigation_Need'] == 'Medium')]
s = oof_preds_binary_2 == 1
l = s[s].index.to_list()
oof_final.loc[l, 'Irrigation_Need'] = 'High'
oof_final = oof_final.drop(columns=['high']).join(oof_preds_2[['high']], how='left')
oof_final['high'] = oof_final['high'].fillna(0.0)

test_final = preds_final.copy()
test_final['Irrigation_Need'] = 'Medium'
test_final.loc[test_preds_binary == 1, 'Irrigation_Need'] = 'Low'
test_final[(test_final['Irrigation_Need'] == 'Medium')]
s = test_preds_binary_2 == 1
l = s[s].index.to_list()
test_final.loc[l, 'Irrigation_Need'] = 'High'
test_final = test_final.drop(columns=['high']).join(preds_final_2[['high']], how='left')
test_final['high'] = test_final['high'].fillna(0.0)

recalls = recall_score(y, oof_final['Irrigation_Need'].map(target_mapping).values, average=None)

print(f"Recall Low:    {recalls[0]:.4f}")
print(f"Recall Medium: {recalls[1]:.4f}")
print(f"Recall High:   {recalls[2]:.4f}")

bal_acc = recalls.mean()
print(f"Balanced Accuracy: {bal_acc:.5f}")

oof_next_2 = oof_final['Irrigation_Need'].map(target_mapping).values
test_next_2 = test_final['Irrigation_Need'].map(target_mapping).values

import datetime

ts = datetime.datetime.now().strftime("%Y%m%d_%H-%M-%S")

sub = test_df[['id']].copy()
sub['Irrigation_Need'] = pd.Series(test_next_2).map(reverse_mapping)
sub.to_csv(f'submission_2_{ts}.csv', index=False)

In [ ]:
oof_final = oof_preds[['id', 'low']].join(oof_preds_2[['high']])
oof_final = oof_final.rename(columns={'high': 'model_2'})
oof_final['medium'] = (1 - oof_final['low']) * (1 - oof_final['model_2'])
oof_final['high'] = (1 - oof_final['low']) * oof_final['model_2']
oof_final = oof_final[['id', 'low', 'medium', 'high']].copy()
oof_final[['low', 'medium', 'high']].copy().to_csv('oof_final.csv', index=False)

test_final = preds_final[['id', 'low']].join(preds_final_2[['high']])
test_final = test_final.rename(columns={'high': 'model_2'})
test_final['medium'] = (1 - test_final['low']) * (1 - test_final['model_2'])
test_final['high'] = (1 - test_final['low']) * test_final['model_2']
test_final = test_final[['id', 'low', 'medium', 'high']].copy()
test_final[['low', 'medium', 'high']].copy().to_csv('test_final.csv', index=False)

In [ ]:
def final(preds, best_a=None, best_b=None):
    if best_a is None or best_b is None:
        n = len(preds)
        mean_high = target.mean()['high']
        mean_low = target.mean()['low']
        idx_high_sorted = preds['high'].argsort().values[::-1]
        low_scores = preds['low'].values
        best_acc = -1
        best_a, best_b = 0, 0
        range_a = np.arange(0, 0.01, 0.0002)
        range_b = np.arange(0, 0.01, 0.0002)
        for a in tqdm(range_a):
            num_high = int(round((mean_high + a) * n))
            high_indices = idx_high_sorted[:num_high]
            is_high = np.zeros(n, dtype=bool)
            is_high[high_indices] = True
            for b in range_b:
                current_preds = np.ones(n, dtype=int) 
                current_preds[high_indices] = 2
                num_low = int(round((mean_low + b) * n))
                temp_low_scores = low_scores.copy()
                temp_low_scores[is_high] = -1e9 
                low_indices = np.argpartition(temp_low_scores, -num_low)[-num_low:]
                current_preds[low_indices] = 0
                score = balanced_accuracy_score(y, current_preds)
                if score > best_acc:
                    best_acc = score
                    best_a, best_b = a, b
                
        print(f'BEST {best_a:.6f} {best_b:.6f} {best_acc:.6f}')
    oof_preds = preds[['id', 'low', 'medium', 'high']].copy()
    oof_preds['Irrigation_Need'] = 'Medium'
    oof_preds = oof_preds.sort_values(by='high')
    oof_preds.iloc[-round((target.mean()['high'] + best_a)*oof_preds.shape[0]):, -1] = 'High'
    oof_preds = oof_preds.sort_values(by='low')
    remaining_mask = oof_preds['Irrigation_Need'] != 'High'
    n_low = int(round((target.mean()['low'] + best_b) * len(oof_preds)))
    low_indices = oof_preds[remaining_mask].nlargest(n_low, 'low').index
    oof_preds.loc[low_indices, 'Irrigation_Need'] = 'Low'
    oof_preds = oof_preds.sort_values(by='id')
    return oof_preds['Irrigation_Need'].map(target_mapping).values, best_a, best_b

In [ ]:
oof_next, best_a, best_b = final(oof_final)

In [ ]:
test_next, _, __ = final(test_final, best_a, best_b)

In [ ]:
recalls = recall_score(y, oof_next, average=None)

print(f"Recall Low:    {recalls[0]:.4f}")
print(f"Recall Medium: {recalls[1]:.4f}")
print(f"Recall High:   {recalls[2]:.4f}")

bal_acc = recalls.mean()
print(f"Balanced Accuracy: {bal_acc:.5f}")

In [ ]:
pd.Series(oof_next).value_counts()

In [ ]:
pd.Series(test_next).value_counts()

In [ ]:
cm = confusion_matrix(y, oof_next)
cm

In [ ]:
pd.Series(test_next_2).value_counts()

In [ ]:
import datetime

ts = datetime.datetime.now().strftime("%Y%m%d_%H-%M-%S")

sub = test_df[['id']].copy()
sub['Irrigation_Need'] = pd.Series(test_next).map(reverse_mapping)
sub.to_csv(f'submission_{ts}.csv', index=False)

In [ ]:
sub#['Irrigation_Need'].value_counts()

In [ ]:
# 